In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
arr = np.array(Image.open("me.png"))

In [ ]:
arr = arr[20:920, :900, :]
arr = arr.min(axis=-1)
arr = 255 - arr

In [ ]:
plt.hist(arr)

In [ ]:
Image.fromarray(arr)

In [ ]:
(arr < 125).sum() / (arr > 125).sum() 

In [ ]:
import torch.nn.functional as F

device = "mps"
# net = nn.Linear(2, 2*K, bias=True)
net = nn.Sequential(
    nn.Linear(2, 40, bias=True),
    nn.ReLU(),
    nn.Linear(40, 40, bias=True),
    nn.ReLU(),
    nn.Linear(40, 1, bias=True),
    nn.Sigmoid()
).to(device)

nn.init.xavier_uniform_(net[0].weight)
nn.init.xavier_uniform_(net[2].weight)
with torch.no_grad():
    net[0].bias.zero_()
    net[2].bias.zero_()

optimizer = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=0)

x = torch.linspace(-1, 1, 900)
y = torch.linspace(-1, 1, 900)
xx, yy = torch.meshgrid(x, y)

data = torch.stack([xx, yy], axis=2).to(device)
truth = (torch.tensor(arr) / 255).to(device)
data = data.view(-1, 2)
truth = truth.view(-1)

HW = data.shape[0]
batch = 10000

In [ ]:
import tqdm
pbar = tqdm.trange(50000)
for i in pbar:
    idx = torch.randint(HW, size=(batch,), device=device)
    xy = data[idx]
    y_true = truth[idx]
    binary_true = (y_true > 1/2).float().unsqueeze(1)
    # mask = (y_true < 1/2) * 1/6 + (y_true > 1/2) * 1
    pred = net(xy)
    # loss = F.binary_cross_entropy(pred, y_true.float().unsqueeze(-1))
    # mask = (y_true < 0.5) * 0.5 + (y_true > 0.5) * 1.0
    loss = F.binary_cross_entropy(pred, y_true.float().unsqueeze(-1), reduction='mean')
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 200 == 0:
        pbar.set_description(f"loss: {loss.item():.6f}")

In [ ]:
with torch.no_grad():
    pred = net(data).detach().cpu()
    image = (pred.clamp(0, 1)*255).to(torch.uint8).view(900, 900).numpy()
    # pred = alpha * F.leaky_relu(z_pos).sum(axis=-1) + F.leaky_relu(z_neg).sum(axis=-1)

In [ ]:
import matplotlib.pyplot as plt

_ = plt.hist(pred.numpy())

In [ ]:
pred.mean()

In [ ]:
Image.fromarray(image, mode="L")

In [ ]:
division = []
with open("activation.txt", "w") as f:
    f.write("[\n")
    for (a, b), c, mul in zip(net[0].weight.tolist(), net[0].bias.tolist(), net[2].weight[0].tolist()):
        a *= mul
        b *= mul
        c *= mul
        f.write(f"({a}, {b}, {c}),\n")
        division.append((a, b, c))
    f.write("]")

In [ ]:
from visualize import *
plot_relu_partition(division)

In [ ]:
render_relu(division)

In [ ]:
how_many_closed_sub_space(division)